# Lab Assignment 10 — NLP using Python-II
**UCS420 Cognitive Computing**

Solutions for Questions 1–5. Each question is organized in its own section.

## Q1. Text preprocessing, tokenization, stopword removal, and word frequency

In [ ]:
import re
import nltk
from collections import Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

text = (
    "Artificial intelligence is changing the way people solve problems. "
    "It helps computers learn from data and recognize useful patterns. "
    "AI tools can support education, healthcare, and scientific research. "
    "Students can use these tools to understand difficult concepts. "
    "However, people should check AI-generated information carefully. "
    "Responsible use of technology can make learning more effective."
)

lower_text = text.lower()
clean_text = re.sub(r"[^a-z0-9\s.]", "", lower_text)

print("Original paragraph:\n", text)
print("\nLowercase text without punctuation (periods retained for sentence splitting):\n", clean_text)

sentences = sent_tokenize(text)
print("\nSentence tokens:")
print(sentences)

print("\nPython split() tokens:")
print(text.split())

print("\nNLTK word_tokenize() tokens:")
print(word_tokenize(text))

stop_words = set(stopwords.words("english"))
words = re.findall(r"\b[a-z]+\b", clean_text)
filtered_words = [word for word in words if word not in stop_words]

print("\nWords after stopword removal:")
print(filtered_words)

frequency = Counter(filtered_words)
print("\nWord frequency distribution (excluding stopwords):")
print(frequency.most_common())

## Q2. Alphabetic words, stemming, and lemmatization

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

alphabetic_words = re.findall(r"\b[a-zA-Z]+\b", text.lower())
words_without_stops = [word for word in alphabetic_words if word not in stop_words]

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed_words = [stemmer.stem(word) for word in words_without_stops]
lemmatized_words = [lemmatizer.lemmatize(word) for word in words_without_stops]

print("Alphabetic words:")
print(alphabetic_words)
print("\nAfter stopword removal:")
print(words_without_stops)
print("\nStemmed words:")
print(stemmed_words)
print("\nLemmatized words:")
print(lemmatized_words)

print("\nComparison:")
print("Stemming cuts words down to forms that may not be real words (for example, 'changing' may become 'chang').")
print("Lemmatization usually returns a meaningful dictionary form (for example, 'patterns' becomes 'pattern').")
print("Use stemming when speed and broad matching are priorities; prefer lemmatization when readable, linguistically meaningful words matter.")

## Q3. Bag of Words and TF-IDF keywords for three texts

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

texts = [
    "Electric vehicles reduce tailpipe pollution and can lower transport emissions.",
    "Solar energy produces clean electricity from sunlight and supports renewable power.",
    "Artificial intelligence helps computers analyze data, recognize patterns, and automate tasks."
]

count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(texts)
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=["Text 1", "Text 2", "Text 3"]
)

print("Bag of Words representation:")
display(bow_df)

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
terms = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=terms,
    index=["Text 1", "Text 2", "Text 3"]
)

print("\nTF-IDF scores:")
display(tfidf_df.round(3))

for i, row in enumerate(tfidf_matrix.toarray()):
    top_indices = row.argsort()[::-1][:3]
    top_keywords = [(terms[j], round(float(row[j]), 3)) for j in top_indices]
    print(f"Top 3 keywords for Text {i + 1}: {top_keywords}")

print("\nInterpretation: TF-IDF gives higher weight to terms that are important in one text but less common across the full collection.")

## Q4. Jaccard and cosine similarity between two technology descriptions

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ai_text = (
    "Artificial intelligence uses algorithms to learn from data. "
    "It can recognize patterns, make predictions, and automate tasks. "
    "AI is used in healthcare, education, and business."
)

blockchain_text = (
    "Blockchain stores records in linked blocks across a distributed network. "
    "It supports transparent transactions and makes records difficult to alter. "
    "Blockchain is used in finance, supply chains, and digital identity."
)

def preprocess_and_tokenize(value):
    value = re.sub(r"[^a-zA-Z\s]", "", value.lower())
    return value.split()

ai_tokens = preprocess_and_tokenize(ai_text)
blockchain_tokens = preprocess_and_tokenize(blockchain_text)

print("AI tokens:")
print(ai_tokens)
print("\nBlockchain tokens:")
print(blockchain_tokens)

ai_set = set(ai_tokens)
blockchain_set = set(blockchain_tokens)
union = ai_set | blockchain_set
intersection = ai_set & blockchain_set
jaccard = len(intersection) / len(union) if union else 0.0

vectorizer = TfidfVectorizer()
tfidf_values = vectorizer.fit_transform([ai_text, blockchain_text])
cosine = float(cosine_similarity(tfidf_values[0:1], tfidf_values[1:2])[0, 0])

print(f"\nShared unique words: {sorted(intersection)}")
print(f"Jaccard similarity: {jaccard:.3f}")
print(f"Cosine similarity (TF-IDF): {cosine:.3f}")

print("\nAnalysis:")
print("Jaccard similarity compares the overlap between unique word sets and ignores word frequency.")
print("TF-IDF cosine similarity compares weighted word vectors, giving less influence to words common across the texts.")
print("For these technology descriptions, cosine similarity can provide a more informative comparison because it accounts for term importance; both scores may be low because the technologies use different specialist vocabulary.")

## Q5. Review sentiment analysis and word cloud for positive reviews

In [ ]:
from textblob import TextBlob
from wordcloud import WordCloud
import matplotlib.pyplot as plt

reviews = [
    "The headphones have excellent sound quality and a comfortable fit. I love the battery life.",
    "This phone is fast, reliable, and easy to use. The camera is fantastic.",
    "The delivery was late and the package arrived damaged. I am very disappointed.",
    "The app is frustrating and crashes often. Customer support was not helpful.",
    "The product is okay. It works as expected but nothing special.",
    "The service was average and the experience was neither good nor bad."
]

results = []
positive_reviews = []

for review in reviews:
    sentiment = TextBlob(review).sentiment
    polarity = sentiment.polarity
    subjectivity = sentiment.subjectivity

    if polarity > 0:
        label = "Positive"
        positive_reviews.append(review)
    elif polarity < 0:
        label = "Negative"
    else:
        label = "Neutral"

    results.append({
        "Review": review,
        "Polarity": round(polarity, 3),
        "Subjectivity": round(subjectivity, 3),
        "Classification": label
    })

sentiment_df = pd.DataFrame(results)
display(sentiment_df)

print("Positive reviews used for the word cloud:")
for review in positive_reviews:
    print("-", review)

if positive_reviews:
    positive_text = " ".join(positive_reviews)
    wordcloud = WordCloud(
        width=900,
        height=450,
        background_color="white",
        collocations=False
    ).generate(positive_text)

    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title("Word Cloud of Positive Reviews")
    plt.show()
else:
    print("No positive reviews were identified, so a word cloud could not be created.")

print("\nPolarity ranges from -1 (negative) to +1 (positive). Subjectivity ranges from 0 (more objective) to 1 (more subjective).")